# DanceTrack: all trackers, default params, both state estimators

In [1]:
import os
import inspect
from collections import defaultdict

import numpy as np
import pandas as pd
import supervision as sv

from trackers import SORTTracker, OCSORTTracker, ByteTrackTracker
from trackers.eval import evaluate_mot_sequences
from trackers.utils.state_representations import XYXYStateEstimator, XCYCSRStateEstimator

DANCETRACK_DET_ROOT = "dancetrack_yolox_dets"
DANCETRACK_GT_ROOT = os.path.join("TrackEval", "data", "gt", "dancetrack")


def build_dets_index(det_list):
    dets_by_frame = defaultdict(list)
    for line in det_list:
        frame_id = int(line.split(",")[0])
        dets_by_frame[frame_id].append(line)
    return dets_by_frame


def get_detections_from_dict(frame_id, dets_by_frame):
    dets = []
    for line in dets_by_frame.get(frame_id, []):
        det = line.split(",")
        x1, y1, x2, y2, conf = float(det[1]), float(det[2]), float(det[3]), float(det[4]), float(det[5])
        dets.append([x1, y1, x2, y2, conf])
    return dets


def make_tracker(tracker_name, state_estimator_class):
    cls_map = {
        "SORT": SORTTracker,
        "OCSORT": OCSORTTracker,
        "ByteTrack": ByteTrackTracker,
    }
    tracker_cls = cls_map[tracker_name]
    kwargs = {}
    try:
        params = inspect.signature(tracker_cls.__init__).parameters
        if "state_estimator_class" in params:
            kwargs["state_estimator_class"] = state_estimator_class
    except Exception:
        pass
    return tracker_cls(**kwargs)


def run_tracker_dancetrack(tracker_name, estimator_name, estimator_class, eval_set="val"):
    tracker = make_tracker(tracker_name, estimator_class)
    outputs_root = f"{tracker_name}_outputs_dancetrack_default_estimators"
    save_dir = os.path.join(outputs_root, f"{eval_set}_{estimator_name.lower()}")
    os.makedirs(save_dir, exist_ok=True)

    det_root = os.path.join(DANCETRACK_DET_ROOT, eval_set)
    gt_dir = os.path.join(DANCETRACK_GT_ROOT, eval_set)
    seqmap = os.path.join(DANCETRACK_GT_ROOT, f"DanceTrack-{eval_set}.txt")

    for seq in sorted(os.listdir(det_root)):
        if not seq.endswith(".txt"):
            continue
        tracker.reset()
        seq_name = seq.split(".")[0]

        with open(os.path.join(det_root, seq), "r") as f_det:
            det_list = f_det.readlines()
            dets_by_frame = build_dets_index(det_list)

        last_frame = int(det_list[-1].split(",")[0])
        output_lines = []
        for frame_id in range(1, last_frame + 1):
            raw_dets = get_detections_from_dict(frame_id, dets_by_frame)
            if raw_dets:
                raw_dets = np.array(raw_dets)
                dets = sv.Detections(xyxy=raw_dets[:, :4], confidence=raw_dets[:, 4])
            else:
                dets = sv.Detections.empty()

            dets = tracker.update(detections=dets)
            for tid, (left, top, right, bottom) in zip(dets.tracker_id, dets.xyxy):
                if tid == -1:
                    continue
                width = right - left
                height = bottom - top
                output_lines.append(
                    f"{frame_id},{int(tid)},{round(left,1)},{round(top,1)},{round(width,1)},{round(height,1)},-1,-1,-1,-1\n"
                )

        with open(os.path.join(save_dir, seq_name + ".txt"), "w") as f:
            f.writelines(output_lines)

    result = evaluate_mot_sequences(
        gt_dir=gt_dir,
        tracker_dir=save_dir,
        metrics=["CLEAR", "HOTA", "Identity"],
        seqmap=seqmap,
    )
    return {
        "tracker": tracker_name,
        "state_estimator": estimator_name,
        "HOTA": result.to_dict()["aggregate"]["HOTA"]["HOTA"],
        "IDF1": result.to_dict()["aggregate"]["Identity"]["IDF1"],
        "MOTA": result.to_dict()["aggregate"]["CLEAR"]["MOTA"],
        "output_dir": save_dir,
    }

In [2]:
estimators = [
    ("XYXY", XYXYStateEstimator),
    ("XCYCSR", XCYCSRStateEstimator),
]
trackers = ["SORT", "OCSORT", "ByteTrack"]

rows = []
for tracker_name in trackers:
    for estimator_name, estimator_class in estimators:
        print(f"Running {tracker_name} with {estimator_name}...")
        rows.append(run_tracker_dancetrack(tracker_name, estimator_name, estimator_class, eval_set="val"))

comparison_df = pd.DataFrame(rows).sort_values(["tracker", "state_estimator"]).reset_index(drop=True)
comparison_df

Running SORT with XYXY...
Running SORT with XCYCSR...
Running OCSORT with XYXY...
Running OCSORT with XCYCSR...
Running ByteTrack with XYXY...
Running ByteTrack with XCYCSR...


,tracker,state_estimator,HOTA,IDF1,MOTA,output_dir
0,ByteTrack,XCYCSR,0.495107,0.488160,0.858289,ByteTrack_outputs_dancetrack_default_estimator...
1,ByteTrack,XYXY,0.501938,0.498866,0.861247,ByteTrack_outputs_dancetrack_default_estimator...
2,OCSORT,XCYCSR,0.517674,0.509338,0.872941,OCSORT_outputs_dancetrack_default_estimators/v...
3,OCSORT,XYXY,0.520006,0.513313,0.874389,OCSORT_outputs_dancetrack_default_estimators/v...
4,SORT,XCYCSR,0.445980,0.406852,0.780749,SORT_outputs_dancetrack_default_estimators/val...
5,SORT,XYXY,0.449949,0.389768,0.805728,SORT_outputs_dancetrack_default_estimators/val...
